# Stage 0.1: TherapistQA preprocessing

Source file: `kaggle_cognitive_distortion_detection_dataset.csv` (2,530 records).

**Output:** `processed/therapistqa.csv` with cleaned text, character-offset spans for the distorted part, and metadata columns.

**Pipeline:**

1. Load and verify columns.
2. Deduplicate: for duplicate patient questions, keep the record with an actual distortion label (drop the `No Distortion` twin).
3. Normalise whitespace.
4. Map `distorted_part` to character offsets in `patient_question` (exact / fuzzy / failed).
5. Filter by length. Inspect long records manually.
6. Handle non-distorted records: keep with NaN spans.
7. Keep `Secondary Distortion` for later ablation.
8. Output the processed CSV.

In [19]:
import difflib
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Project paths — adjust if the notebook lives elsewhere.
DATASETS_DIR = Path('./datasets/')
OUTPUT_DIR = Path('processed')
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCE_CSV = DATASETS_DIR / 'kaggle_cognitive_distortion_detection_dataset.csv'
OUTPUT_CSV = OUTPUT_DIR / 'therapistqa.csv'

# Decision log: drop/fuzzy match/length filter gets appended here so the
# decisions are reproducible and reportable in Section 3.6 of the thesis.
decision_log = []

def log(msg: str) -> None:
    decision_log.append(msg)
    print(msg)

## Step 1: Load and verify

In [20]:
df_raw = pd.read_csv(SOURCE_CSV)
log(f'Loaded {len(df_raw)} records from {SOURCE_CSV.name}')

expected_columns = [
    'Id_Number',
    'Patient Question',
    'Distorted part',
    'Dominant Distortion',
    'Secondary Distortion (Optional)',
]
assert list(df_raw.columns) == expected_columns, (
    f'Unexpected columns: {list(df_raw.columns)}'
)

df_raw.head(3)

Loaded 2530 records from kaggle_cognitive_distortion_detection_dataset.csv


,Id_Number,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,4500,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,4501,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,4502,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN


In [21]:
# Rename to snake_case internally.
df = df_raw.rename(
    columns={
        'Id_Number': 'id',
        'Patient Question': 'patient_question',
        'Distorted part': 'distorted_part',
        'Dominant Distortion': 'dominant_distortion',
        'Secondary Distortion (Optional)': 'secondary_distortion',
    }
).copy()

log('Dominant distortion value counts:')
print(df['dominant_distortion'].value_counts(dropna=False).to_string())

Dominant distortion value counts:
dominant_distortion
No Distortion              933
Mind Reading               239
Overgeneralization         239
Magnification              195
Labeling                   165
Personalization            153
Fortune-telling            143
Emotional Reasoning        134
Mental filter              122
Should statements          107
All-or-nothing thinking    100


## Step 2: Deduplicate

One patient question appears twice. When one copy is labelled `No Distortion` and the other is labelled with a real distortion, the real distortion label is more informative, so the `No Distortion` copy is dropped.

In [22]:
# Inspect duplicates by patient_question.
dup_mask = df['patient_question'].duplicated(keep=False)
dup_df = df[dup_mask].sort_values('patient_question')
log(f'Duplicate records (by patient_question): {len(dup_df)}')

if len(dup_df) > 0:
    display_cols = ['id', 'dominant_distortion', 'secondary_distortion']
    # Show each duplicate group.
    for pq, group in dup_df.groupby('patient_question'):
        print('---')
        print(f'patient_question (first 120 chars): {pq[:120]!r}')
        print(group[display_cols].to_string(index=False))

Duplicate records (by patient_question): 2
---
patient_question (first 120 chars): 'I grew up in a upper middle class family. I was given most things I wanted and all the essential things a child would ne'
 id dominant_distortion secondary_distortion
265       No Distortion                  NaN
307  Overgeneralization                  NaN


In [23]:
# Drop rows where the same patient_question appears with 'No Distortion' AND a
# real distortion. Keep the real distortion.
drop_ids = []
for pq, group in df[dup_mask].groupby('patient_question'):
    labels = group['dominant_distortion'].tolist()
    has_nondistorted = 'No Distortion' in labels
    has_distorted = any(l != 'No Distortion' for l in labels)
    if has_nondistorted and has_distorted:
        to_drop = group[group['dominant_distortion'] == 'No Distortion']
        drop_ids.extend(to_drop['id'].tolist())

log(f'Dropping {len(drop_ids)} "No Distortion" twins of distorted records')
log(f'Dropped ids: {drop_ids}')

df = df[~df['id'].isin(drop_ids)].reset_index(drop=True)
log(f'Records remaining after deduplication: {len(df)}')

# Any duplicates that survive (e.g. two different real distortions) get flagged.
remaining_dup_mask = df['patient_question'].duplicated(keep=False)
log(f'Surviving duplicate records (two different distortion labels on same '
    f'question, kept with flag): {remaining_dup_mask.sum()}')

Dropping 1 "No Distortion" twins of distorted records
Dropped ids: [265]
Records remaining after deduplication: 2529
Surviving duplicate records (two different distortion labels on same question, kept with flag): 0


## Step 3: Normalise whitespace

Collapse runs of whitespace, normalise line breaks to `\n`, strip leading/trailing whitespace. Applied identically to both `patient_question` and `distorted_part` so the span search in Step 4 compares like with like.

In [24]:
def normalise_whitespace(text):
    if pd.isna(text):
        return text
    # Normalise CR / CRLF to LF.
    text = str(text).replace('\r\n', '\n').replace('\r', '\n')
    # Collapse runs of non-newline whitespace (tabs, multiple spaces) to one space.
    text = re.sub(r'[^\S\n]+', ' ', text)
    # Collapse runs of blank lines to a single newline.
    text = re.sub(r'\n+', '\n', text)
    # Trim spaces around newlines.
    text = re.sub(r' *\n *', '\n', text)
    return text.strip()

df['patient_question'] = df['patient_question'].map(normalise_whitespace)
df['distorted_part'] = df['distorted_part'].map(normalise_whitespace)

# Quick sanity check.
sample = df[df['distorted_part'].notna()].iloc[0]
print('patient_question:')
print(repr(sample['patient_question'][:]))
print('\ndistorted_part:')
print(repr(sample['distorted_part'][:]))

patient_question:
'Hello, I have a beautiful,smart,outgoing and amazing five year old little girl. Yesterday she came to me and said mom can you take me to the doctor. I ask her what was wrong and she replied: I hear voices in my ears but I dont see the people saying it. She says it happened during school doing a reading circle. She thought someone called her stupid and let the teacher know. The teacher said no one said anything. It happened again when my husband was talking to my other children, she said I heard daddy say shut up, but he didnt really say it. The voice are always fimilar (someone she knows) Im very concerned about this and hope it has nothing to do with my pregnancy while on active duty.'

distorted_part:
'The voice are always fimilar (someone she knows) Im very concerned about this and hope it has nothing to do with my pregnancy while on active duty.'


## Step 4: Map `distorted_part` to character offsets

For each record, locate the distorted span inside the patient question. The `span_match_type` column carries this label forward.

In [25]:
def locate_span(patient_question, distorted_part):
    """Return (start, end, match_type). start/end are None if failed."""
    if pd.isna(distorted_part) or distorted_part == '':
        return None, None, None  # No span to locate.

    idx = patient_question.find(distorted_part)
    if idx != -1:
        return idx, idx + len(distorted_part), 'exact'
    return None, None, 'failed'

starts, ends, types = [], [], []
for _, row in df.iterrows():
    s, e, t = locate_span(row['patient_question'], row['distorted_part'])
    starts.append(s)
    ends.append(e)
    types.append(t)

df['distorted_part_start_char'] = starts
df['distorted_part_end_char'] = ends
df['span_match_type'] = types

log('span_match_type distribution (includes NaN for non-distorted records):')
print(df['span_match_type'].value_counts(dropna=False).to_string())

span_match_type distribution (includes NaN for non-distorted records):
span_match_type
exact     1385
None       932
failed     212


In [26]:
# Inspect failed matches.
failed_df = df[df['span_match_type'] == 'failed']
log(f'Failed span matches: {len(failed_df)}')

if len(failed_df) > 0:
    for _, row in failed_df.head(3).iterrows():
        print('---')
        print(f'id={row["id"]}, distortion={row["dominant_distortion"]}')
        print(f'patient_question: {row["patient_question"][:]!r}\n')
        print(f'distorted_part: {row["distorted_part"][:]!r}')

Failed span matches: 212
---
id=4501, distortion=Labeling
patient_question: 'Since I was about 16 years old I’ve had these “spells” where it’s impossible for me to control my emotions. They come on suddenly and with absolutely no triggers whatsoever, although for hours to days before they happen I’ll sometimes have uncomfortably high levels of energy–I’ll feel either obnoxious and silly (but in a kind of desperate way, if that makes sense) or I’ll be really productive and jittery. Both of these high energy feelings are unsettling, maybe because I normally almost never have high levels of energy. The “spells,” “episodes,” whatever they are, involve extreme guilt and disgust with myself–but it’s a very visceral feeling and it’s overwhelming. I feel trapped inside my disgusting self and like I’m willing to do anything to escape it. I feel incredibly agitated and yet trapped at the same time–like I want to move or do something but I can’t. I also just feel extremely empty and sad–like, I’m

## Multi-span recovery for concatenated annotations

Inspection of the failures shows that the annotators often concatenate two or more non-contiguous sentences into a single `distorted_part` string. Example: the annotated span contains `"...escape it.I'm just afraid..."` while the two sentences are several paragraphs apart in the original question. A single-window search cannot recover these.

The fix: split the annotated `distorted_part` on sentence boundaries and locate each piece independently. A record is considered `multi_span` if two or more non-overlapping pieces are found, and the character-offset columns then hold the *first* piece's offsets (for backwards compatibility with single-span consumers) while a new `all_spans` column stores the full list of `(start, end)` tuples as a JSON string.

In [27]:
import json

# Simple sentence splitter — punctuation followed by a capital letter, or by a
# lowercase letter glued to the previous sentence without a space (common here,
# e.g. "escape it.I'm just afraid..."). Keeping it minimal to avoid pulling in
# nltk/spacy at preprocessing time.
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])(?=[A-Z])|(?<=[.!?])\s+')

def split_sentences(text, min_len=15):
    """Split into sentences, dropping pieces shorter than min_len."""
    pieces = _SENT_SPLIT_RE.split(text)
    return [p.strip() for p in pieces if len(p.strip()) >= min_len]


def locate_multi_span(patient_question, distorted_part):
    """Try to locate every sentence of distorted_part inside patient_question.

    Returns (spans, match_type) where spans is a list of (start, end) tuples.
    """
    sentences = split_sentences(distorted_part)
    if len(sentences) < 2:
        return [], 'not_multi'  # nothing to split, leave as failed

    spans = []
    for sent in sentences:
        idx = patient_question.find(sent)
        if idx != -1:
            spans.append((idx, idx + len(sent)))
            continue
        # Fall back to fuzzy for this piece.
        matcher = difflib.SequenceMatcher(None, patient_question, sent,
                                          autojunk=False)
        m = matcher.find_longest_match(0, len(patient_question), 0, len(sent))
        if m.size == 0 or m.size < 0.8 * len(sent):
            continue
        spans.append((m.a, m.a + m.size))

    if len(spans) >= 2:
        # Merge overlapping or adjacent spans (≤ MERGE_GAP chars apart — catches
        # touching sentences separated by punctuation or a single space).
        spans.sort()
        MERGE_GAP = 3
        merged = [spans[0]]
        for s, e in spans[1:]:
            if s <= merged[-1][1] + MERGE_GAP:
                merged[-1] = (merged[-1][0], max(merged[-1][1], e))
            else:
                merged.append((s, e))
        return merged, 'multi_span' if len(merged) >= 2 else 'exact'

    if len(spans) == 1:
        return spans, 'partial'  # only one of several pieces was found

    return [], 'failed'


# Rerun locate for records currently marked 'failed'.
df['all_spans'] = df.apply(
    lambda r: json.dumps([[int(r['distorted_part_start_char']),
                           int(r['distorted_part_end_char'])]])
    if pd.notna(r['distorted_part_start_char']) else None,
    axis=1,
)

failed_idx = df[df['span_match_type'] == 'failed'].index
recovered_multi = 0
recovered_partial = 0
recovered_exact = 0
for idx in failed_idx:
    row = df.loc[idx]
    spans, match_type = locate_multi_span(row['patient_question'],
                                          row['distorted_part'])
    if not spans:
        # keep as 'failed'
        continue

    df.at[idx, 'distorted_part_start_char'] = spans[0][0]
    df.at[idx, 'distorted_part_end_char'] = spans[0][1]
    df.at[idx, 'span_match_type'] = match_type
    df.at[idx, 'all_spans'] = json.dumps([[s, e] for s, e in spans])

    if match_type == 'multi_span':
        recovered_multi += 1
    elif match_type == 'partial':
        recovered_partial += 1
    else:
        recovered_exact += 1

log(f'Multi-span recovery: {recovered_multi} records recovered as multi_span')
log(f'Partial recovery:    {recovered_partial} records recovered as partial '
    f'(only some sentences found)')

log('\nUpdated span_match_type distribution:')
print(df['span_match_type'].value_counts(dropna=False).to_string())

Multi-span recovery: 203 records recovered as multi_span
Partial recovery:    0 records recovered as partial (only some sentences found)

Updated span_match_type distribution:
span_match_type
exact         1391
None           932
multi_span     203
failed           3


In [28]:
DROP_RECORDS = {
    160: 'distorted_part belongs to a different patient question',
    887: 'distorted_part is a paraphrase not grounded in the question',
    1536: 'distorted_part is a paraphrase not grounded in the question',
}

before = len(df)
df = df[~df['id'].isin(DROP_RECORDS.keys())]
dropped = before - len(df)
log(f'Dropped {dropped} records due to annotation errors:')
for record_id, reason in DROP_RECORDS.items():
    log(f'  id={record_id}: {reason}')

Dropped 3 records due to annotation errors:
  id=160: distorted_part belongs to a different patient question
  id=887: distorted_part is a paraphrase not grounded in the question
  id=1536: distorted_part is a paraphrase not grounded in the question


In [29]:
# Spot-check two or three recovered multi_span records.
multi_df = df[df['span_match_type'] == 'multi_span']
for _, row in multi_df.head(3).iterrows():
    print('---')
    print(f'id={row["id"]}')
    print(f'annotated distorted_part: {row["distorted_part"][:]!r}')
    spans = json.loads(row['all_spans'])
    for i, (s, e) in enumerate(spans):
        print(f'  span {i}: [{s}:{e}] -> {row["patient_question"][s:e][:]!r}')

---
id=4501
annotated distorted_part: 'I feel trapped inside my disgusting self and like I’m willing to do anything to escape it.I’m just afraid sometimes that since I’m not really thinking during these times that I might do something to myself–especially since I think during these times that I don’t deserve to get help or that I’m not worth disturbing people by calling them.'
  span 0: [673:763] -> 'I feel trapped inside my disgusting self and like I’m willing to do anything to escape it.'
  span 1: [1457:1706] -> 'I’m just afraid sometimes that since I’m not really thinking during these times that I might do something to myself–especially since I think during these times that I don’t deserve to get help or that I’m not worth disturbing people by calling them.'
---
id=4504
annotated distorted_part: 'I refused to go because I didn’t know if it was going to go on my record which could possibly affect my dreams of becoming a psychiatrist.I have a major fear of choking and drowning.It see

## Step 5: Length statistics and filtering

DeBERTa-v3-base has a 512-token window ~ 2,000–2,500 characters, so the truncation is the realistic outcome for the rare very-long records.

The length distribution is computed below and the long-tail records are displayed for manual inspection.

In [30]:
df['patient_question_length'] = df['patient_question'].str.len()
df['distorted_part_length'] = df['distorted_part'].str.len()

log('patient_question_length summary:')
print(df['patient_question_length'].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

log('\ndistorted_part_length summary (non-null only):')
print(df['distorted_part_length'].dropna().describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

patient_question_length summary:
count    2526.000000
mean      870.294537
std       593.264683
min       250.000000
50%       654.500000
75%      1159.750000
90%      1840.000000
95%      2053.750000
99%      2455.000000
max      5657.000000

distorted_part_length summary (non-null only):
count    1594.000000
mean      188.547679
std       150.615667
min        11.000000
50%       151.000000
75%       228.750000
90%       349.000000
95%       443.000000
99%       796.420000
max      1824.000000


In [31]:
# How many records fall into each length bucket?
for threshold in [1000, 1500, 2000, 2500, 3000, 4000, 5000]:
    n_over = (df['patient_question_length'] > threshold).sum()
    pct = 100 * n_over / len(df)
    log(f'  patient_question_length > {threshold:>5}: '
        f'{n_over:>4} records ({pct:.2f}%)')

log(f'\n  patient_question_length < 20: '
    f'{(df["patient_question_length"] < 20).sum()} records')

  patient_question_length >  1000:  766 records (30.32%)
  patient_question_length >  1500:  394 records (15.60%)
  patient_question_length >  2000:  166 records (6.57%)
  patient_question_length >  2500:   24 records (0.95%)
  patient_question_length >  3000:   11 records (0.44%)
  patient_question_length >  4000:    4 records (0.16%)
  patient_question_length >  5000:    2 records (0.08%)

  patient_question_length < 20: 0 records


In [32]:
# Surface the long records so they can be inspected one by one and trimmed.
# With DeBERTa-v3's 512-token window (~2,000-2,500 chars for English),
# the practical threshold to worry about is ~2,500 chars.
# Truncation works; the question is whether the distorted
# span survives inside the truncation window.
INSPECT_THRESHOLD = 2500
long_df = df[df['patient_question_length'] > INSPECT_THRESHOLD].copy()
long_df = long_df.sort_values('patient_question_length', ascending=False)
log(f'Records longer than {INSPECT_THRESHOLD} chars: {len(long_df)}')

# For each long record, compute whether the distorted span sits within the
# first 2,000 characters -> if yes, truncation loses nothing important.
def span_fits_in_window(row, window_chars=2000):
    if pd.isna(row['distorted_part_end_char']):
        return None
    return row['distorted_part_end_char'] <= window_chars

long_df['span_fits_2000'] = long_df.apply(span_fits_in_window, axis=1)
display(long_df[['id', 'patient_question_length', 'distorted_part_start_char',
                 'distorted_part_end_char', 'span_fits_2000',
                 'dominant_distortion']]
        .sort_values('span_fits_2000', ascending=True, na_position='last'))

Records longer than 2500 chars: 24


,id,patient_question_length,distorted_part_start_char,distorted_part_end_char,span_fits_2000,dominant_distortion
1740,1659,5657,2905.0,2985.0,False,Mental filter
1848,1767,5069,3026.0,3161.0,False,Mind Reading
1760,1679,4382,3669.0,3741.0,False,All-or-nothing thinking
1894,1813,4254,4124.0,4239.0,False,Personalization
531,333,2573,2364.0,2478.0,False,Fortune-telling
193,4696,3787,3038.0,3297.0,False,Mind Reading
429,230,2577,2455.0,2532.0,False,Personalization
182,4685,3342,2595.0,2796.0,False,Personalization
1919,1838,2710,2321.0,2558.0,False,Should statements
402,203,2877,1953.0,2094.0,False,Magnification


In [33]:
# Apply the actual filtering. Current plan:
#   - Keep everything up to INSPECT_THRESHOLD chars untouched.
#   - Records > INSPECT_THRESHOLD chars: keep them but flag them in 'length_flag' so the
#     next stage can decide whether to truncate, hand-trim, or skip.

df['length_flag'] = np.where(
    df['patient_question_length'] > INSPECT_THRESHOLD,
    'over_threshold',
    'ok',
)
log(f'Records flagged length_flag=over_threshold: '
    f'{(df["length_flag"] == "over_threshold").sum()}')

Records flagged length_flag=over_threshold: 24


## Step 6: Handle non-distorted records

Records with `Dominant Distortion == 'No Distortion'` have no `Distorted part`, so their span columns stay `NaN`. They are retained for classification (as the negative class) but can be excluded from span experiments via the `is_distorted` flag.

In [34]:
df['is_distorted'] = df['dominant_distortion'] != 'No Distortion'

log(f'is_distorted=True : {df["is_distorted"].sum()}')
log(f'is_distorted=False: {(~df["is_distorted"]).sum()}')

# Sanity: non-distorted records should not have a distorted_part.
leak = df[(~df['is_distorted']) & df['distorted_part'].notna()]
log(f'Non-distorted records that still have a distorted_part (should be 0): {len(leak)}')
if len(leak) > 0:
    print(leak[['id', 'dominant_distortion', 'distorted_part']].head())

is_distorted=True : 1594
is_distorted=False: 932
Non-distorted records that still have a distorted_part (should be 0): 0


## Step 8: Final schema and output

Columns in `processed/therapistqa.csv`:

| Column | Purpose |
|---|---|
| `id` | Original Kaggle `Id_Number`. |
| `patient_question` | Whitespace-normalised patient text. |
| `distorted_part` | Whitespace-normalised highlighted span. `NaN` for non-distorted. |
| `distorted_part_start_char` | Character offset into `patient_question`. `NaN` for non-distorted. |
| `distorted_part_end_char` | Character offset (exclusive) of the first matched piece. `NaN` for non-distorted. |
| `all_spans` | JSON-encoded list of `[start, end]` pairs. Singleton for `exact`, multiple for `multi_span`/`partial`. |
| `span_match_type` | `exact`, `multi_span`, `partial`, `failed`, or `NaN`. Flags span reliability for BIO tagging in Stage 3. |
| `dominant_distortion` | Primary label. |
| `secondary_distortion` | Optional second label. Used only for the multi-label ablation (Stage 2 optional). |
| `is_distorted` | Boolean. Excludes `No Distortion` from span experiments with a single filter. |
| `patient_question_length` | Character count. |
| `distorted_part_length` | Character count of the span. |
| `length_flag` | `ok` or `over_threshold`. Flags records that may need hand-trimming before transformer input. |

In [35]:
output_columns = [
    'id',
    'patient_question',
    'distorted_part',
    'distorted_part_start_char',
    'distorted_part_end_char',
    'all_spans',
    'span_match_type',
    'dominant_distortion',
    'secondary_distortion',
    'is_distorted',
    'patient_question_length',
    'distorted_part_length',
    'length_flag',
]

df_out = df[output_columns].copy()
df_out.to_csv(OUTPUT_CSV, index=False)
log(f'\nWrote {len(df_out)} records to {OUTPUT_CSV}')

df_out.head(3)


Wrote 2526 records to processed/therapistqa.csv


,id,patient_question,distorted_part,distorted_part_start_char,distorted_part_end_char,all_spans,span_match_type,dominant_distortion,secondary_distortion,is_distorted,patient_question_length,distorted_part_length,length_flag
0,4500,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,547.0,694.0,"[[547, 694]]",exact,Personalization,NaN,True,694,147.0,ok
1,4501,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,673.0,763.0,"[[673, 763], [1457, 1706]]",multi_span,Labeling,Emotional Reasoning,True,1706,339.0,ok
2,4502,So I’ve been dating on and off this guy for al...,NaN,NaN,NaN,None,None,No Distortion,NaN,False,2155,NaN,ok


In [36]:
log('\n=== Summary ===')
log(f'Total records:           {len(df_out)}')
log(f'Distorted:               {df_out["is_distorted"].sum()}')
log(f'Non-distorted:           {(~df_out["is_distorted"]).sum()}')
log(f'Span match - exact:      {(df_out["span_match_type"] == "exact").sum()}')
log(f'Span match - multi_span: {(df_out["span_match_type"] == "multi_span").sum()}')
log(f'Span match - partial:    {(df_out["span_match_type"] == "partial").sum()}')
log(f'Span match - failed:     {(df_out["span_match_type"] == "failed").sum()}')
log(f'Over threshold ch:       {(df_out["length_flag"] == "over_threshold").sum()}')

log('\nLabel distribution:')
print(df_out['dominant_distortion'].value_counts().to_string())


=== Summary ===
Total records:           2526
Distorted:               1594
Non-distorted:           932
Span match - exact:      1391
Span match - multi_span: 203
Span match - partial:    0
Span match - failed:     0
Over threshold ch:       24

Label distribution:
dominant_distortion
No Distortion              932
Mind Reading               239
Overgeneralization         239
Magnification              194
Labeling                   165
Personalization            152
Fortune-telling            143
Emotional Reasoning        134
Mental filter              122
Should statements          107
All-or-nothing thinking     99


In [37]:
# Persist the decision log
decision_log_path = OUTPUT_DIR / 'therapistqa_decision_log.txt'
decision_log_path.write_text('\n'.join(decision_log), encoding='utf-8')
print(f'Decision log written to {decision_log_path}')

Decision log written to processed/therapistqa_decision_log.txt
